# chart.json to CSV + Visualization

Converts `data/sc/json/mb_chart.json` into a CSV file under `data/sc/csv/` and
renders an interactive price chart using Plotly.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

# Resolve paths relative to the repo root (parent of this notebook's `scripts` dir)
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
JSON_PATH = REPO_ROOT / "data" / "sc" / "json" / "mb_chart.json"
CSV_DIR = REPO_ROOT / "data" / "sc" / "csv"
CSV_PATH = CSV_DIR / "mb_chart.csv"

JSON_PATH, CSV_PATH

In [ ]:
with JSON_PATH.open("r", encoding="utf-8") as f:
    payload = json.load(f)

chart = payload["data"]
isin = chart["isin"]
currency = chart["currency"]
source = chart["source"]
timeframe = chart["timeframe"]

df = pd.DataFrame(chart["data_points"])
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"])
df = df.sort_values("timestamp_utc").reset_index(drop=True)
df.insert(0, "isin", isin)

df.head()

In [ ]:
CSV_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(CSV_PATH, index=False)
print(f"Wrote {len(df)} rows to {CSV_PATH}")

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df["timestamp_utc"],
        y=df["mid_price"],
        mode="lines+markers",
        name="Mid price",
    )
)

closing_ref = chart.get("closing_reference_point")
if closing_ref:
    fig.add_trace(
        go.Scatter(
            x=[pd.to_datetime(closing_ref["timestamp_utc"])],
            y=[closing_ref["mid_price"]],
            mode="markers",
            marker=dict(color="red", size=10, symbol="star"),
            name="Closing reference",
        )
    )

fig.update_layout(
    title=f"{isin} mid price ({timeframe}, {source})",
    xaxis_title="Timestamp (UTC)",
    yaxis_title=f"Mid price ({currency})",
    hovermode="x unified",
    template="plotly_white",
)
fig.show()

## quote.json to CSV + Visualization

Converts `data/sc/json/mb_quote.json` (a point-in-time quote snapshot) into two
CSV files under `data/sc/csv/` and renders a performance-by-timeframe bar chart.

In [ ]:
QUOTE_JSON_PATH = REPO_ROOT / "data" / "sc" / "json" / "mb_quote.json"
QUOTE_CSV_PATH = CSV_DIR / "mb_quote.csv"
QUOTE_PERF_CSV_PATH = CSV_DIR / "mb_quote_performances.csv"

with QUOTE_JSON_PATH.open("r", encoding="utf-8") as f:
    quote_payload = json.load(f)

quote = quote_payload["data"]["result"]
performances = quote.pop("quote_performances")

quote_df = pd.DataFrame([quote])
quote_df["quote_timestamp_utc"] = pd.to_datetime(quote_df["quote_timestamp_utc"])

perf_df = pd.DataFrame(performances)
perf_df.insert(0, "isin", quote["isin"])

quote_df

In [ ]:
CSV_DIR.mkdir(parents=True, exist_ok=True)
quote_df.to_csv(QUOTE_CSV_PATH, index=False)
perf_df.to_csv(QUOTE_PERF_CSV_PATH, index=False)
print(f"Wrote quote to {QUOTE_CSV_PATH}")
print(f"Wrote {len(perf_df)} performance rows to {QUOTE_PERF_CSV_PATH}")

In [ ]:
perf_fig = go.Figure()
perf_fig.add_trace(
    go.Bar(
        x=perf_df["timeframe"],
        y=perf_df["performance"],
        marker_color=["crimson" if v < 0 else "seagreen" for v in perf_df["performance"]],
    )
)
perf_fig.update_layout(
    title=f"{quote['name']} ({quote['isin']}) performance by timeframe",
    xaxis_title="Timeframe",
    yaxis_title="Performance (fraction)",
    yaxis_tickformat=".1%",
    template="plotly_white",
)
perf_fig.show()

## holdings.json to CSV + Visualization

Converts `data/sc/json/holdings.json` (current portfolio positions) into a CSV file
under `data/sc/csv/` and renders a portfolio allocation treemap colored by unrealized
profit/loss.

In [ ]:
HOLDINGS_JSON_PATH = REPO_ROOT / "data" / "sc" / "json" / "holdings.json"
HOLDINGS_CSV_PATH = CSV_DIR / "holdings.csv"

with HOLDINGS_JSON_PATH.open("r", encoding="utf-8") as f:
    holdings_payload = json.load(f)

holdings_df = pd.DataFrame(holdings_payload["data"]["result"]["items"])
holdings_df["quote_timestamp_utc"] = pd.to_datetime(holdings_df["quote_timestamp_utc"])
holdings_df["unrealized_pnl"] = (holdings_df["quote_mid_price"] - holdings_df["fifo_price"]) * holdings_df["quantity"]
holdings_df["unrealized_pnl_pct"] = holdings_df["quote_mid_price"] / holdings_df["fifo_price"] - 1
holdings_df["portfolio_weight"] = holdings_df["valuation"] / holdings_df["valuation"].sum()

holdings_df.head()

In [ ]:
CSV_DIR.mkdir(parents=True, exist_ok=True)
holdings_df.to_csv(HOLDINGS_CSV_PATH, index=False)
print(f"Wrote {len(holdings_df)} rows to {HOLDINGS_CSV_PATH}")

In [ ]:
holdings_fig = go.Figure(
    go.Treemap(
        labels=holdings_df["name"],
        parents=[""] * len(holdings_df),
        values=holdings_df["valuation"],
        text=holdings_df["isin"],
        customdata=holdings_df[["quantity", "fifo_price", "quote_mid_price", "unrealized_pnl_pct"]],
        hovertemplate=(
            "%{label}<br>qty=%{customdata[0]}<br>avg cost=%{customdata[1]:.2f}"
            "<br>price=%{customdata[2]:.2f}<br>valuation=%{value:.2f}"
            "<br>P&L=%{customdata[3]:.1%}<extra></extra>"
        ),
        marker=dict(
            colors=holdings_df["unrealized_pnl_pct"],
            colorscale="RdYlGn",
            # Cap the range at +/-50% so a single outlier (e.g. AMD's +323%) doesn't
            # wash out the color contrast for the rest of the portfolio
            cmin=-0.5,
            cmax=0.5,
            colorbar=dict(title="P&L %", tickformat=".0%"),
        ),
    )
)
holdings_fig.update_layout(
    title="Portfolio holdings: allocation (size) and unrealized P&L (color)",
    template="plotly_white",
)
holdings_fig.show()

## transactions.json to CSV + Visualization

Converts `data/sc/json/transactions.json` (cash deposits and withdrawals) into a
CSV file under `data/sc/csv/` and renders a timeline bar chart plus a cumulative
cash-flow line chart.

In [ ]:
TXN_JSON_PATH = REPO_ROOT / "data" / "sc" / "json" / "transactions.json"
TXN_CSV_PATH = CSV_DIR / "transactions.csv"

with TXN_JSON_PATH.open("r", encoding="utf-8") as f:
    txn_payload = json.load(f)

txn_df = pd.DataFrame(txn_payload["transactions"])
txn_df["last_event_datetime"] = pd.to_datetime(txn_df["last_event_datetime"])
txn_df = txn_df.sort_values("last_event_datetime").reset_index(drop=True)

# Drop columns that are constant or not useful for analysis
txn_df = txn_df.drop(columns=["documents", "unknown_summary_type", "summary_type"])

txn_df

In [ ]:
CSV_DIR.mkdir(parents=True, exist_ok=True)
txn_df.to_csv(TXN_CSV_PATH, index=False)
print(f"Wrote {len(txn_df)} rows to {TXN_CSV_PATH}")

In [ ]:
# Bar chart: individual deposits and withdrawals over time
deposit_mask = txn_df["cash_transaction_type"] == "DEPOSIT"
deposit_subset = txn_df[deposit_mask]
withdrawal_mask = txn_df["cash_transaction_type"] == "WITHDRAWAL"
withdrawal_subset = txn_df[withdrawal_mask]

# Bar width in milliseconds: ~10 days wide so bars are visible on a multi-year axis
BAR_WIDTH_MS = 10 * 24 * 60 * 60 * 1000

# Create figure with explicit data
txn_fig = go.Figure()

# Add deposits
txn_fig.add_trace(
    go.Bar(
        x=deposit_subset["last_event_datetime"],
        y=deposit_subset["amount"],
        name="Deposit",
        marker_color="seagreen",
        width=BAR_WIDTH_MS,
        hovertemplate="%{x|%Y-%m-%d}<br>%{y:,.0f} EUR<extra></extra>",
    )
)

# Add withdrawals
txn_fig.add_trace(
    go.Bar(
        x=withdrawal_subset["last_event_datetime"],
        y=withdrawal_subset["amount"],
        name="Withdrawal",
        marker_color="crimson",
        width=BAR_WIDTH_MS,
        hovertemplate="%{x|%Y-%m-%d}<br>%{y:,.0f} EUR<extra></extra>",
    )
)

print(f"\nFigure has {len(txn_fig.data)} traces")
print(f"Trace 0 (deposits): {len(txn_fig.data[0].x)} points")
print(f"Trace 1 (withdrawals): {len(txn_fig.data[1].x)} points")

txn_fig.update_layout(
    title="Cash transactions over time",
    xaxis_title="Date",
    yaxis_title="Amount (EUR)",
    barmode="overlay",   # overlay so both traces render at their own dates
    template="plotly_white",
    hovermode="x unified",
    height=600,
    bargap=0,
)
txn_fig.show()

In [ ]:
# Cumulative cash flow over time
txn_sorted = txn_df.sort_values("last_event_datetime").copy()
txn_sorted["cumulative"] = txn_sorted["amount"].cumsum()

cum_fig = go.Figure()
cum_fig.add_trace(
    go.Scatter(
        x=txn_sorted["last_event_datetime"],
        y=txn_sorted["cumulative"],
        mode="lines+markers",
        name="Cumulative net cash flow",
        line=dict(color="royalblue"),
        hovertemplate="%{x|%Y-%m-%d}<br>%{y:,.0f} EUR<extra></extra>",
    )
)
cum_fig.add_hline(y=0, line_dash="dash", line_color="gray")
cum_fig.update_layout(
    title="Cumulative net cash flow (deposits - withdrawals)",
    xaxis_title="Date",
    yaxis_title="Cumulative amount (EUR)",
    template="plotly_white",
    hovermode="x unified",
)
cum_fig.show()